# NCF model using merge data Taobao with M5

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

!pip install -q tensorflow tensorflow-recommenders


In [3]:
# ===== MUST BE FIRST CELL =====
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

!pip uninstall -y keras tensorflow tensorflow-recommenders
!pip install tensorflow tensorflow-recommenders pandas

print("✅ Restart runtime after this cell")


Found existing installation: keras 3.13.2
Uninstalling keras-3.13.2:
  Successfully uninstalled keras-3.13.2
Found existing installation: tensorflow 2.19.1
Uninstalling tensorflow-2.19.1:
  Successfully uninstalled tensorflow-2.19.1
Found existing installation: tensorflow-recommenders 0.7.7
Uninstalling tensorflow-recommenders-0.7.7:
  Successfully uninstalled tensorflow-recommenders-0.7.7
  Using cached tensorflow-2.21.0-cp312-cp312-manylinux_2_27_x86_64.whl.metadata (4.4 kB)
  Using cached tensorflow_recommenders-0.7.7-py3-none-any.whl.metadata (4.6 kB)
  Using cached protobuf-7.34.0-cp310-abi3-manylinux2014_x86_64.whl.metadata (595 bytes)
  Using cached keras-3.13.2-py3-none-any.whl.metadata (6.3 kB)
  Using cached tensorflow-2.19.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
Using cached tensorflow_recommenders-0.7.7-py3-none-any.whl (108 kB)
Using cached keras-3.13.2-py3-none-any.whl (1.5 MB)
Using cached tensorflow-2.19.1-cp312-cp312-manylinux_2_1

In [4]:
import os

BASE_DIR = "/content/drive/MyDrive/ncf_project_new"

os.makedirs(f"{BASE_DIR}/training", exist_ok=True)
os.makedirs(f"{BASE_DIR}/models/ncf_model", exist_ok=True)

print("Project folders created")

Project folders created


In [5]:
# -----------------------
# Load dataset
# -----------------------
import pandas as pd

# Load CSV file
df = pd.read_csv('/content/drive/MyDrive/datasets/last_merge_new.csv')

# Sort by timestamp
df = df.sort_values("timestamp").reset_index(drop=True)

# Select only needed columns
df = df[['user_id', 'behavior_type', 'timestamp', 'item_id', 'category_id']]

In [6]:
import pandas as pd
import numpy as np
import tensorflow as tf
import tensorflow_recommenders as tfrs
import pickle
from typing import Dict, Text

print("✅ TensorFlow version:", tf.__version__)
print("✅ TFRS imported successfully")


✅ TensorFlow version: 2.19.1
✅ TFRS imported successfully


In [7]:
import numpy as np
import tensorflow as tf
import tensorflow_recommenders as tfrs
from typing import Dict, Text

# -----------------------
# Split train/test
# -----------------------
train_df = df.iloc[:1_800_000].copy()
test_df  = df.iloc[1_800_000:].copy()

# Map behavior type to numeric weight
behavior_map = {"pv": 1.0, "fav": 3.0, "cart": 5.0, "buy": 10.0}
train_df["weight"] = train_df["behavior_type"].map(behavior_map).astype(float)
test_df["weight"]  = test_df["behavior_type"].map(behavior_map).astype(float)

# Ensure string type
for col in ["user_id", "item_id", "category_id"]:
    train_df[col] = train_df[col].astype(str)
    test_df[col]  = test_df[col].astype(str)

# Save test set
test_df.to_csv("test_set_200k_new.csv", index=False)

# -----------------------
# Split train into chunks
# -----------------------
chunk_size = 100_000
num_chunks = int(np.ceil(len(train_df) / chunk_size))

for i in range(num_chunks):
    train_df.iloc[i*chunk_size:(i+1)*chunk_size] \
        .to_csv(f"train_chunk_{i+1}.csv", index=False)

# -----------------------
# Vocabulary
# -----------------------
unique_users = train_df["user_id"].unique().tolist()
unique_items = train_df["item_id"].unique().tolist()

# -----------------------
# NCF Model
# -----------------------
class NCFModel(tf.keras.Model):
    def __init__(self, unique_users, unique_items, embedding_dim=32):
        super().__init__()

        self.user_embedding = tf.keras.Sequential([
            tf.keras.layers.StringLookup(
                vocabulary=unique_users, mask_token=None
            ),
            tf.keras.layers.Embedding(len(unique_users) + 1, embedding_dim)
        ])

        self.item_embedding = tf.keras.Sequential([
            tf.keras.layers.StringLookup(
                vocabulary=unique_items, mask_token=None
            ),
            tf.keras.layers.Embedding(len(unique_items) + 1, embedding_dim)
        ])

        self.mlp = tf.keras.Sequential([
            tf.keras.layers.Dense(64, activation="relu"),
            tf.keras.layers.Dense(32, activation="relu"),
            tf.keras.layers.Dense(1)
        ])

    def call(self, inputs, training=False):
        user_emb = self.user_embedding(inputs["user_id"])
        item_emb = self.item_embedding(inputs["item_id"])
        x = tf.concat([user_emb, item_emb], axis=1)
        return self.mlp(x)

# -----------------------
# TFRS Wrapper
# -----------------------
class NCFRecommender(tfrs.models.Model):
    def __init__(self, unique_users, unique_items):
        super().__init__()
        self.ncf = NCFModel(unique_users, unique_items)
        self.task = tfrs.tasks.Ranking(
            loss=tf.keras.losses.MeanSquaredError(),
            metrics=[tf.keras.metrics.RootMeanSquaredError()]
        )

    def compute_loss(
        self, features: Dict[Text, tf.Tensor], training=False
    ) -> tf.Tensor:
        labels = features["weight"]
        predictions = self.ncf(
            {
                "user_id": features["user_id"],
                "item_id": features["item_id"]
            },
            training=training
        )
        return self.task(labels=labels, predictions=predictions)

# -----------------------
# Initialize
# -----------------------
model = NCFRecommender(unique_users, unique_items)
model.compile(optimizer=tf.keras.optimizers.Adam(0.001))

In [8]:
# -----------------------
# Chunk-by-chunk training
# -----------------------
for i in range(1, num_chunks + 1):
    print(f"\n=== Training chunk {i}/{num_chunks} ===")

    chunk = pd.read_csv(f"train_chunk_{i}.csv")

    # Ensure correct type
    for col in ["user_id", "item_id"]:
        chunk[col] = chunk[col].astype(str)

    ds = tf.data.Dataset.from_tensor_slices({
        "user_id": chunk["user_id"].values,
        "item_id": chunk["item_id"].values,
        "weight": chunk["weight"].values.astype(np.float32)
    })

    ds = ds.shuffle(50_000).batch(2048).prefetch(tf.data.AUTOTUNE)

    model.fit(ds, epochs=3 if i == 1 else 1, verbose=1)


# -----------------------
# Evaluate
# -----------------------
test_df = pd.read_csv("test_set_200k_new.csv")

test_ds = tf.data.Dataset.from_tensor_slices({
    "user_id": test_df["user_id"].astype(str).values,
    "item_id": test_df["item_id"].astype(str).values,
    "weight": test_df["weight"].astype(np.float32).values
}).batch(2048).prefetch(tf.data.AUTOTUNE)

results = model.evaluate(test_ds, verbose=1)
print("\n✅ Test results:", results)


=== Training chunk 1/18 ===
Epoch 1/3
49/49 [==============================] - 4s 39ms/step - root_mean_squared_error: 1.9166 - loss: 3.6370 - regularization_loss: 0.0000e+00 - total_loss: 3.6370
Epoch 2/3
49/49 [==============================] - 3s 62ms/step - root_mean_squared_error: 1.5621 - loss: 2.4368 - regularization_loss: 0.0000e+00 - total_loss: 2.4368
Epoch 3/3
49/49 [==============================] - 2s 41ms/step - root_mean_squared_error: 1.4918 - loss: 2.2318 - regularization_loss: 0.0000e+00 - total_loss: 2.2318

=== Training chunk 2/18 ===
49/49 [==============================] - 2s 38ms/step - root_mean_squared_error: 1.5041 - loss: 2.2601 - regularization_loss: 0.0000e+00 - total_loss: 2.2601

=== Training chunk 3/18 ===
49/49 [==============================] - 2s 37ms/step - root_mean_squared_error: 1.4994 - loss: 2.2373 - regularization_loss: 0.0000e+00 - total_loss: 2.2373

=== Training chunk 4/18 ===
49/49 [==============================] - 2s 41ms/step - root_mea

In [9]:
def recommend_top_k_ncf(user_id, unique_items, model, k=10, batch_size=4096):
    """
    Recommend Top-K items for a given user using trained NCF model
    """

    user_id = str(user_id)
    items = np.array(unique_items).astype(str)

    scores = []

    # Process in batches (important for large item sets)
    for i in range(0, len(items), batch_size):
        batch_items = items[i:i + batch_size]

        # Repeat user_id for each item
        user_batch = np.repeat(user_id, len(batch_items))

        # Predict scores
        batch_scores = model.ncf({
            "user_id": tf.constant(user_batch),
            "item_id": tf.constant(batch_items)
        })

        scores.append(batch_scores.numpy().reshape(-1))

    # Combine all scores
    scores = np.concatenate(scores)

    # Get top-k indices
    top_k_idx = np.argsort(scores)[-k:][::-1]

    # Print results
    print(f"\nTop-{k} recommendations for User {user_id}:")
    for rank, idx in enumerate(top_k_idx, 1):
        print(f"{rank}. Item ID: {items[idx]} | Score: {scores[idx]:.4f}")

    return items[top_k_idx], scores[top_k_idx]

In [10]:
recommend_top_k_ncf(
    user_id=train_df["user_id"].iloc[0],
    unique_items=unique_items,
    model=model,
    k=10
)


Top-10 recommendations for User 170980:
1. Item ID: FOODS_3_515 | Score: 1.6764
2. Item ID: FOODS_3_624 | Score: 1.6583
3. Item ID: HOUSEHOLD_2_373 | Score: 1.6564
4. Item ID: HOBBIES_1_394 | Score: 1.6413
5. Item ID: FOODS_3_214 | Score: 1.6208
6. Item ID: HOUSEHOLD_1_385 | Score: 1.6193
7. Item ID: FOODS_2_051 | Score: 1.6183
8. Item ID: FOODS_2_264 | Score: 1.6146
9. Item ID: FOODS_1_178 | Score: 1.6121
10. Item ID: HOUSEHOLD_1_260 | Score: 1.6089


(array(['FOODS_3_515', 'FOODS_3_624', 'HOUSEHOLD_2_373', 'HOBBIES_1_394',
        'FOODS_3_214', 'HOUSEHOLD_1_385', 'FOODS_2_051', 'FOODS_2_264',
        'FOODS_1_178', 'HOUSEHOLD_1_260'], dtype='<U15'),
 array([1.6764339, 1.6582661, 1.6564021, 1.6412661, 1.6208489, 1.6192887,
        1.6183162, 1.6146237, 1.6121428, 1.608932 ], dtype=float32))

In [11]:
import tensorflow as tf
import os

EXPORT_DIR = f"{BASE_DIR}/models/ncf_savedmodel_new/1"

# Clean old export (important)
import shutil
if os.path.exists(EXPORT_DIR):
    shutil.rmtree(EXPORT_DIR)

class NCFServing(tf.Module):
    def __init__(self, trained_model):
        super().__init__()
        self.model = trained_model.ncf  # your NCFModel

    @tf.function(input_signature=[
        tf.TensorSpec(shape=[None], dtype=tf.string, name="user_id"),
        tf.TensorSpec(shape=[None], dtype=tf.string, name="item_id"),
    ])
    def serving_default(self, user_id, item_id):
        scores = self.model({"user_id": user_id, "item_id": item_id})  # (batch,1)
        return {"score": tf.squeeze(scores, axis=1)}  # (batch,)

serving = NCFServing(model)

tf.saved_model.save(
    serving,
    EXPORT_DIR,
    signatures={"serving_default": serving.serving_default}
)

print("✅ Re-exported SavedModel with named inputs to:", EXPORT_DIR)


✅ Re-exported SavedModel with named inputs to: /content/drive/MyDrive/ncf_project_new/models/ncf_savedmodel_new/1


In [12]:
import shutil

ZIP_PATH = f"{BASE_DIR}/models/ncf_savedmodel_new.zip"
shutil.make_archive(ZIP_PATH.replace(".zip",""), "zip", f"{BASE_DIR}/models/ncf_savedmodel_new")
print("✅ Zipped to:", ZIP_PATH)


✅ Zipped to: /content/drive/MyDrive/ncf_project_new/models/ncf_savedmodel_new.zip


# Deep matrix factorizaion

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

!pip install -q tensorflow tensorflow-recommenders


In [15]:
# ===== MUST BE FIRST CELL =====
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

!pip uninstall -y keras tensorflow tensorflow-recommenders
!pip install tensorflow tensorflow-recommenders pandas

print("✅ Restart runtime after this cell")


Found existing installation: keras 3.13.2
Uninstalling keras-3.13.2:
  Successfully uninstalled keras-3.13.2
Found existing installation: tensorflow 2.19.1
Uninstalling tensorflow-2.19.1:
  Successfully uninstalled tensorflow-2.19.1
Found existing installation: tensorflow-recommenders 0.7.7
Uninstalling tensorflow-recommenders-0.7.7:
  Successfully uninstalled tensorflow-recommenders-0.7.7
  Using cached tensorflow-2.21.0-cp312-cp312-manylinux_2_27_x86_64.whl.metadata (4.4 kB)
  Using cached tensorflow_recommenders-0.7.7-py3-none-any.whl.metadata (4.6 kB)
  Using cached protobuf-7.34.0-cp310-abi3-manylinux2014_x86_64.whl.metadata (595 bytes)
  Using cached keras-3.13.2-py3-none-any.whl.metadata (6.3 kB)
  Using cached tensorflow-2.19.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
Using cached tensorflow_recommenders-0.7.7-py3-none-any.whl (108 kB)
Using cached keras-3.13.2-py3-none-any.whl (1.5 MB)
Using cached tensorflow-2.19.1-cp312-cp312-manylinux_2_1

✅ Restart runtime after this cell


In [16]:
import os

BASE_DIR = "/content/drive/MyDrive/dmf_project_new"

os.makedirs(f"{BASE_DIR}/training", exist_ok=True)
os.makedirs(f"{BASE_DIR}/models/dmf_model", exist_ok=True)

print("Project folders created")

Project folders created


In [17]:
import pandas as pd
import numpy as np
import tensorflow as tf
import tensorflow_recommenders as tfrs
from typing import Dict, Text

df_deep = pd.read_csv('/content/drive/MyDrive/datasets/last_merge_new.csv')

In [18]:
print("Total unique users:", df_deep["user_id"].nunique())
print("Total unique items:", df_deep["item_id"].nunique())

behavior_count = df_deep["behavior_type"].value_counts().reset_index()
behavior_count.columns = ["behavior_type", "count"]
print(behavior_count)


Total unique users: 19544
Total unique items: 3049
  behavior_type    count
0            pv  1791216
1          cart   111015
2           fav    57526
3           buy    40243


In [19]:
# Preprocessing
df_deep = df_deep.sort_values("timestamp").reset_index(drop=True)

df_deep = df_deep[['user_id', 'item_id', 'category_id', 'behavior_type','timestamp']]

In [20]:
# Training/test splits
train_df = df_deep.iloc[:1_800_000].copy()
test_df  = df_deep.iloc[1_800_000:].copy()

In [21]:
# Map behavior to weight
behavior_map = {"pv": 1.0, "fav": 3.0, "cart": 5.0, "buy": 10.0}

train_df["weight"] = train_df["behavior_type"].map(behavior_map)
test_df["weight"]  = test_df["behavior_type"].map(behavior_map)

In [22]:
# Convert to string
for col in ["user_id", "item_id", "category_id"]:
    train_df[col] = train_df[col].astype(str)
    test_df[col]  = test_df[col].astype(str)

In [23]:
# Save test set
test_df.to_csv("test_set_200k.csv", index=False)

In [24]:
# Split train into chunks (100,000 rows each)
chunk_size = 100_000
num_chunks = int(np.ceil(len(train_df) / chunk_size))

for i in range(num_chunks):
    chunk = train_df.iloc[i*chunk_size:(i+1)*chunk_size]
    chunk.to_csv(f"train_chunk_{i+1}.csv", index=False)

In [25]:
# Unique vocab
unique_users = train_df["user_id"].unique()
unique_items = train_df["item_id"].unique()


In [26]:
# DMF model
class DMFModel(tf.keras.Model):
    def __init__(self, unique_users, unique_items, embedding_dim=32, hidden_units=[64, 32]):
        super().__init__()

        self.user_embedding = tf.keras.Sequential([
            tf.keras.layers.StringLookup(vocabulary=unique_users),
            tf.keras.layers.Embedding(len(unique_users) + 1, embedding_dim)
        ])

        self.item_embedding = tf.keras.Sequential([
            tf.keras.layers.StringLookup(vocabulary=unique_items),
            tf.keras.layers.Embedding(len(unique_items) + 1, embedding_dim)
        ])

        self.user_mlp = tf.keras.Sequential()
        self.item_mlp = tf.keras.Sequential()

        for units in hidden_units:
            self.user_mlp.add(tf.keras.layers.Dense(units, activation='relu'))
            self.item_mlp.add(tf.keras.layers.Dense(units, activation='relu'))

        self.output_layer = tf.keras.layers.Dense(1)

    def call(self, inputs):
        user_emb = self.user_mlp(self.user_embedding(inputs["user_id"]))
        item_emb = self.item_mlp(self.item_embedding(inputs["item_id"]))
        x = tf.multiply(user_emb, item_emb)
        return self.output_layer(x)

In [27]:
# Recommender model
class DMFRecommender(tfrs.models.Model):
    def __init__(self, unique_users, unique_items):
        super().__init__()
        self.dmf_model = DMFModel(unique_users, unique_items)

        self.task = tfrs.tasks.Ranking(
            loss=tf.keras.losses.MeanSquaredError(),
            metrics=[tf.keras.metrics.RootMeanSquaredError()]
        )

    def compute_loss(self, features: Dict[Text, tf.Tensor], training=False):
        features_copy = features.copy()
        labels = features_copy.pop("weight")
        predictions = self.dmf_model(features_copy)
        return self.task(labels=labels, predictions=predictions)

In [28]:
# Initialize model
model = DMFRecommender(unique_users, unique_items)
model.compile(optimizer=tf.keras.optimizers.Adam(0.001))


In [29]:
# Train chunk by chunk
chunk_files = [f"train_chunk_{i+1}.csv" for i in range(num_chunks)]

for i, chunk_file in enumerate(chunk_files, 1):
    print(f"\n=== Training on chunk {i} ===")
    chunk = pd.read_csv(chunk_file)

    for col in ["user_id", "item_id", "category_id"]:
        chunk[col] = chunk[col].astype(str)

    train_ds_chunk = tf.data.Dataset.from_tensor_slices({
        "user_id": chunk["user_id"].values,
        "item_id": chunk["item_id"].values,
        "weight": chunk["weight"].astype(float).values
    }).shuffle(len(chunk)).batch(2048).prefetch(tf.data.AUTOTUNE)

    epochs = 3 if i == 1 else 1
    model.fit(train_ds_chunk, epochs=epochs, verbose=1)


=== Training on chunk 1 ===
Epoch 1/3
49/49 [==============================] - 3s 36ms/step - root_mean_squared_error: 2.0850 - loss: 4.3094 - regularization_loss: 0.0000e+00 - total_loss: 4.3094
Epoch 2/3
49/49 [==============================] - 2s 36ms/step - root_mean_squared_error: 1.6231 - loss: 2.6302 - regularization_loss: 0.0000e+00 - total_loss: 2.6302
Epoch 3/3
49/49 [==============================] - 1s 20ms/step - root_mean_squared_error: 1.5072 - loss: 2.2730 - regularization_loss: 0.0000e+00 - total_loss: 2.2730

=== Training on chunk 2 ===
49/49 [==============================] - 1s 23ms/step - root_mean_squared_error: 1.5060 - loss: 2.2548 - regularization_loss: 0.0000e+00 - total_loss: 2.2548

=== Training on chunk 3 ===
49/49 [==============================] - 1s 20ms/step - root_mean_squared_error: 1.4997 - loss: 2.2407 - regularization_loss: 0.0000e+00 - total_loss: 2.2407

=== Training on chunk 4 ===
49/49 [==============================] - 1s 21ms/step - root_mea

In [30]:
# Load test dataset
test_df = pd.read_csv("test_set_200k.csv")

for col in ["user_id", "item_id", "category_id"]:
    test_df[col] = test_df[col].astype(str)

test_ds = tf.data.Dataset.from_tensor_slices({
    "user_id": test_df["user_id"].values,
    "item_id": test_df["item_id"].values,
    "weight": test_df["weight"].astype(float).values
}).batch(2048).prefetch(tf.data.AUTOTUNE)

In [31]:
# Evaluation
results = model.evaluate(test_ds)
print("\nTest evaluation results:", results)

98/98 [==============================] - 1s 6ms/step - root_mean_squared_error: 1.5106 - loss: 2.2827 - regularization_loss: 0.0000e+00 - total_loss: 2.2827

Test evaluation results: [1.5105559825897217, 2.3500239849090576, 0.0, 2.3500239849090576]


In [32]:
# Recommendation Function
def recommend_top_k_dmf(user_id, unique_items, model, k=10, batch_size=4096):
    user_id = str(user_id)
    items = unique_items.astype(str)
    scores = []

    for i in range(0, len(items), batch_size):
        batch_items = items[i:i + batch_size]
        user_batch = np.repeat(user_id, len(batch_items))

        batch_scores = model.dmf_model({
            "user_id": tf.constant(user_batch),
            "item_id": tf.constant(batch_items)
        })

        scores.append(batch_scores.numpy().reshape(-1))

    scores = np.concatenate(scores)
    top_k_idx = np.argsort(scores)[-k:][::-1]

    print(f"\nTop-{k} recommendations for User {user_id}:")
    for rank, idx in enumerate(top_k_idx, 1):
        print(f"{rank}. Item ID: {items[idx]} | Score: {scores[idx]:.2f}")

    return items[top_k_idx], scores[top_k_idx]

In [33]:
# Example recommendation
recommend_top_k_dmf(train_df["user_id"].iloc[0], unique_items, model, k=10)


Top-10 recommendations for User 170980:
1. Item ID: FOODS_3_444 | Score: 1.54
2. Item ID: HOUSEHOLD_1_249 | Score: 1.53
3. Item ID: FOODS_3_335 | Score: 1.53
4. Item ID: FOODS_2_130 | Score: 1.53
5. Item ID: HOUSEHOLD_1_342 | Score: 1.52
6. Item ID: HOUSEHOLD_1_385 | Score: 1.52
7. Item ID: HOBBIES_1_394 | Score: 1.52
8. Item ID: HOUSEHOLD_1_018 | Score: 1.52
9. Item ID: FOODS_3_515 | Score: 1.51
10. Item ID: FOODS_2_189 | Score: 1.51


(array(['FOODS_3_444', 'HOUSEHOLD_1_249', 'FOODS_3_335', 'FOODS_2_130',
        'HOUSEHOLD_1_342', 'HOUSEHOLD_1_385', 'HOBBIES_1_394',
        'HOUSEHOLD_1_018', 'FOODS_3_515', 'FOODS_2_189'], dtype='<U15'),
 array([1.5356534, 1.5347568, 1.5286015, 1.5284934, 1.523271 , 1.521628 ,
        1.5214903, 1.5213445, 1.5131277, 1.5065287], dtype=float32))